# Task 1
## Integrantes
* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007

**Pregunta 1**

El gerente de producto de VisorShelf le presenta la siguiente situación: el sistema detecta una lata de atún en el anaquel y devuelve la caja predicha b̂ = (142, 89, 218, 165) en formato (x_min, y_min, x_max, y_max). El radiólogo de calidad del cliente anota manualmente la caja real b* = (138, 84, 222, 170). El cliente pregunta: "¿Qué tan buena es esa detección?

Con esto en mente, responda las siguientes preguntas en su reporte:


1. **Calcule manualmente el IoU entre las dos cajas. Muestre paso a paso el cálculo del área de intersección, el área de unión y el valor final. Explique en términos no técnicos qué significa ese número para el cliente de VisorShelf.**

Tengo la caja predicha $b̂ = (142, 89, 218, 165)$ y la caja real $b* = (138, 84, 222, 170)$.

Primero calculo la intersección:

- $x_{min} = max(142, 138) = 142$
- $y_{min} = max(89, 84) = 89$
- $x_{max} = min(218, 222) = 218$
- $y_{max} = min(165, 170) = 165$

Entonces me quedaría:

- Ancho de la intersección: $218 - 142 = 76$
- Alto de la intersección: $165 - 89 = 76$
- Área de la intersección: $76 * 76 = 5776$

Ahora calculo las áreas individuales:

- Área predicha $= (218 - 142) * (165 - 89) = 76 * 76 = 5776$
- Área real $= (222 - 138) * (170 - 84) = 84 * 86 = 7224$

Después calculo la unión:

$|U| = 5776 + 7224 - 5776 = 7224$

Finalmente:

$IoU = |I| / |U| = 5776 / 7224 = 0.7996$

Por tanto, el IoU es aproximadamente $0.80$.

Por lo que, yo le diría al cliente que la detección es bastante buena: la caja del sistema coincide casi un $80%$ con la caja correcta. Es decir, el modelo sí encontró bien la lata y la encerró de forma bastante cercana a la realidad, aunque no de manera perfecta.

2. **En la fórmula IoU = |I| / |U|, identifique qué representa cada símbolo (|I|, |U|) y explique por qué el denominador es la unión y no el área del ground truth. ¿Qué problema concreto evita esa decisión de diseño?**

$|I|$ representa el área de intersección, o sea, la parte que comparten la caja predicha y la caja real.

$|U|$ representa el área de unión, o sea, todo el espacio cubierto por ambas cajas juntas, sin contar dos veces la parte superpuesta.

Yo considero que el denominador debe ser la unión y no solo el área del ground truth porque así penalizo dos tipos de error al mismo tiempo: cuando la caja se queda corta y cuando la caja se pasa de grande. Si usara solo el área real como denominador, una caja exageradamente grande que cubra por completo el objeto podría parecer “muy buena”, aunque en realidad esté localizando mal. Esa decisión de diseño evita premiar detecciones infladas que abarcan demasiado fondo o incluso productos vecinos, algo especialmente problemático en anaqueles densos.

3. **El equipo de VisorShelf está evaluando dos umbrales de IoU para decidir si una detección es válida: θ = 0.5 y θ = 0.75. ¿Cuál recomendaría para el sistema de auditoría de anaqueles y por qué? Considere el impacto operativo de los falsos positivos y falsos negativos en el negocio del cliente.**

Yo recomendaría $θ = 0.5$ para el sistema de auditoría de anaqueles.

La razón es porque en VisorShelf me interesa detectar correctamente que el producto está presente, ausente o mal ubicado, pero no necesito una caja perfecta al nivel casi milimétrico en cada frame. Si subo el umbral a $0.75$, vuelvo la validación mucho más estricta y aumento el riesgo de falsos negativos, es decir, cajas razonables que sí detectan el producto pero quedan descartadas por no estar suficientemente ajustadas. Del lado del negocio, eso puede traducirse en reportar quiebres de stock inexistentes o en no reconocer productos que sí están en el anaquel.

Tomo en cuenta que $0.75$ reduce algunos falsos positivos de localización, pero en este caso me parece más costoso perder productos reales que aceptar cajas ligeramente imperfectas. Por eso, para una operación robusta y sensible al inventario, $0.5$ me parece el balance correcto.

**Pregunta 2**

Durante una prueba piloto en una tienda de conveniencia, el detector de VisorShelf analiza una imagen con 15 productos en el anaquel. El modelo genera 18 predicciones. Tras aplicar el umbral IoU = 0.5, el equipo clasifica: 12 TP, 6 FP y 3 FN:

Con base a esto, responda dentro de su reporte:

4. **Calcule la Precisión y el Recall para esta prueba. En las fórmulas P = TP / (TP + FP) y R = TP / (TP + FN), explique verbalmente qué mide cada término del denominador y por qué ambas métricas son necesarias para evaluar el sistema.**

Hago los cálculos:

- $P = TP / (TP + FP) = 12 / (12 + 6) = 12 / 18 = 0.6667$
- $R = TP / (TP + FN) = 12 / (12 + 3) = 12 / 15 = 0.80$

Entonces:

Precisión $= 0.6667$ o $66.67%$
Recall $= 0.80$ o $80%$

Para la explicación de denominadores:

- En $TP + FP$, estoy contando todas las detecciones positivas que emitió el sistema. Por eso la Precisión mide qué tan confiables son mis alertas.
- En $TP + FN$, estoy contando todos los objetos reales que debían detectarse. Por eso el Recall mide qué tanto del anaquel logré cubrir realmente.

Necesito ambas métricas porque pueden contar historias distintas. Un sistema puede ser muy preciso pero dejar pasar muchos productos, o puede detectar casi todo pero generar demasiadas falsas alarmas.

5. **El director de operaciones de la tienda le dice: "Prefiero que el sistema no se pierda ningún quiebre de stock, aunque a veces nos avise de falsos alarmas." Traduzca esa preferencia a términos de Precisión y Recall. ¿Qué umbral de confianza ajustaría y en qué dirección?**

Cuando el director dice que prefiere no perderse ningún quiebre de stock, aunque existan algunas falsas alarmas, yo traduzco eso como una preferencia por alto Recall, incluso si la Precisión baja un poco.

Pienso que me conviene bajar el umbral de confianza $τ$. Y si lo reduzco, dejo pasar más detecciones: suben los verdaderos positivos potenciales, pero también pueden subir los falsos positivos. Aun así, esa dirección tiene sentido porque el costo de no detectar una ausencia real de producto es mayor que el costo de revisar una alarma extra.

6. **Explique qué es el mAP (Mean Average Precision) y por qué es más informativo que reportar un único valor de Precisión o Recall. En su explicación, distinga entre mAP@0.5 (protocolo PASCAL VOC) y mAP@0.5:0.95 (protocolo COCO), y argumente cuál protocolo sería más exigente para VisorShelf y por qué.**

Yo entiendo el mAP como una métrica que resume el desempeño del detector a través de distintos niveles de confianza y, además, considerando la calidad de localización de las cajas. No me da una sola foto fija como Precisión o Recall en un único umbral, sino una visión más completa del comportamiento del modelo.

Diría que es más informativo porque un solo valor de Precisión o Recall depende mucho del punto de operación que yo elija. En cambio, el mAP resume el equilibrio entre ambas métricas a lo largo de la curva Precision-Recall.

La diferencia entre ambos protocolos para mí son:

- $mAP@0.5$ evalúa si la caja se considera correcta cuando el IoU es al menos $0.5$. Es el criterio clásico de PASCAL VOC.
- $mAP@0.5:0.95$ promedia el desempeño sobre varios umbrales de IoU, desde $0.5$ hasta $0.95$. Es el criterio de COCO y es mucho más estricto.

Yo diría que $mAP@0.5:0.95$ es más exigente para VisorShelf, porque en un anaquel denso no basta con “caer más o menos cerca”; también importa que la caja esté bien ajustada y no se coma parte del producto vecino. Justamente por eso ese protocolo refleja mejor la dificultad real del problema.

**Pregunta 3**

En una imagen de anaquel con 40 productos, el modelo de VisorShelf genera 312 cajas candidatas antes de cualquier postprocesamiento. El cliente observa el resultado intermedio y exclama: "¡El sistema está viendo el mismo producto decenas de veces!"

Responda lo siguiente en su reporte:

7. **Explique al cliente qué es el Non-Maximum Suppression (NMS) y por qué el detector genera múltiples cajas para el mismo objeto. Describa el algoritmo paso a paso en lenguaje no técnico.**

Yo le explicaría al cliente que el detector no “ve mal”; más bien evalúa muchísimas posibilidades parecidas alrededor de cada producto, y por eso varias cajas terminan apuntando al mismo objeto.

El Non-Maximum Suppression, o NMS, es el paso que limpia esos duplicados. Funciona así:

1. El sistema ordena todas las cajas por nivel de confianza.
2. Se queda primero con la caja que parece más segura.
3. Luego revisa cuáles otras cajas se parecen demasiado a esa misma caja, es decir, cuáles se superponen mucho.
4. Esas cajas redundantes se eliminan.
5. Después repite el proceso con la siguiente caja más confiable de las que quedaron.

Por lo que, es detector que propone muchas opciones y el NMS se encarga de dejar solo la mejor por producto.

8. **El parámetro $θ_NMS$ controla qué tan agresivo es el NMS al suprimir cajas. En un anaquel densamente poblado donde los productos están uno junto al otro casi sin espacio, ¿qué valor de θ_NMS recomendaría (alto o bajo) y por qué? Argumente el riesgo en cada dirección.**

En un anaquel muy poblado, donde los productos están casi pegados, yo recomendaría un $θ_NMS$ relativamente alto.

La razón es que, si el umbral es muy bajo, el NMS se vuelve demasiado agresivo y puede eliminar cajas que en realidad corresponden a productos distintos pero muy cercanos. Eso aumenta los falsos negativos. En cambio, un umbral más alto tolera más superposición antes de suprimir, lo cual ayuda a conservar productos vecinos legítimos.

El riesgo de cada extremo es claro:

- Si $θ_NMS$ es muy bajo, puedo “fusionar” productos distintos y perder detecciones reales.
- Si $θ_NMS$ es demasiado alto, dejo pasar duplicados y aumento falsos positivos.

Por eso yo no usaría un extremo, pero sí lo pondría más alto que en un escenario con objetos separados.

9. **¿En qué orden se deben aplicar el umbral de confianza $τ$ y el NMS? Justifique la respuesta y explique qué sucedería computacionalmente si se invierte ese orden en un sistema que procesa 30 imágenes por minuto**

Yo aplicaría primero el umbral de confianza $τ$ y después el NMS.

Primero filtro las cajas débiles o claramente poco confiables. Luego, sobre ese conjunto ya reducido, ejecuto NMS para quitar duplicados. Ese orden tiene sentido porque el NMS implica comparar superposiciones entre cajas, y si dejo demasiadas cajas irrelevantes desde el inicio, hago más comparaciones de las necesarias.

Si invierto el orden, obligo al sistema a correr NMS sobre muchas cajas basura. En este caso, partiría de 312 cajas candidatas por imagen, lo que elevaría el costo computacional y empeoraría la latencia.

# Referencias

- GeeksforGeeks. (2025a, April 26). What is NonMaximum Suppression? GeeksforGeeks. https://www.geeksforgeeks.org/computer-vision/what-is-non-maximum-suppression/ 
- GeeksforGeeks. (2025b, July 23). Evaluating Object Detection Models: Methods and Metrics. GeeksforGeeks. https://www.geeksforgeeks.org/evaluating-object-detection-models-methods-and-metrics/ 
- GeeksforGeeks. (2025c, July 23). Mean Average precision (MAP) in computer vision. GeeksforGeeks. https://www.geeksforgeeks.org/mean-average-precision-map-in-computer-vision/ 
- GeeksforGeeks. (2025d, August 2). Precision and recall in machine learning. GeeksforGeeks. https://www.geeksforgeeks.org/precision-and-recall-in-machine-learning/